# Trabajo Práctico — Series Temporales

## 07 — Pronóstico con ensambles de Machine Learning

**Objetivo de este notebook:** comparar varios modelos de ML (LightGBM, XGBoost, Random Forest) para pronosticar cada una de las tres series, usando `skforecast` como framework de pronóstico recursivo con backtesting temporal correcto (evita el error de *data leakage* que la propia cátedra señaló en su ejemplo de stacking), y compararlos contra el SARIMA (`05_sarima.ipynb`) y el VAR (`06_var.ipynb`) ya calculados.

**Por qué skforecast y no una sola pasada de train/test:** un `train_test_split` simple sobrestima qué tan bien generalizaría el modelo en producción. `backtesting_forecaster` repite la validación en varias ventanas temporales sucesivas (como una validación cruzada que respeta el orden temporal), dando una estimación de error más realista.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold, bayesian_search_forecaster

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

datos = pd.read_csv(
    PROJECT_ROOT / "data/processed/serie_california.csv",
    index_col="datetime_utc",
    parse_dates=True,
)
print(f"Filas: {len(datos):,} | Rango: {datos.index.min()} -> {datos.index.max()}")

## 1. Preparación: índice posicional y variables de calendario

`skforecast` exige que el índice tenga una frecuencia regular. Nuestra serie limpia tiene algunos saltos (los huecos largos ya se descartaron en `02_limpieza.ipynb`), así que forzar `asfreq("h")` reintroduciría huecos como `NaN` — la misma trampa de rendimiento que ya nos pasó con SARIMAX en `05_sarima.ipynb`. En cambio, se usa un **índice posicional** (0, 1, 2, ...) para el modelado, y se guarda el índice de fechas real aparte para poder graficar e interpretar resultados con fechas reales.

Como variables exógenas (que sí se conocen de antemano, a diferencia de las otras dos series físicas) se usan variables de calendario: hora del día, día de la semana y mes — el ciclo diario/semanal que ya vimos que domina estas series.

In [ ]:
fechas_reales = datos.index

exog = pd.DataFrame(
    {
        "hora": fechas_reales.hour,
        "dia_semana": fechas_reales.dayofweek,
        "mes": fechas_reales.month,
    },
    index=range(len(datos)),
)

datos_pos = datos.reset_index(drop=True)

HORAS_TEST = 24 * 7
LAGS = [1, 2, 3, 24, 48, 72]

print(f"Variables de calendario (exog): {list(exog.columns)}")
print(f"Lags usados: {LAGS}")
print(f"Horas de test (holdout final): {HORAS_TEST}")

## 2. Búsqueda de hiperparámetros por modelo (Optuna vía `bayesian_search_forecaster`)

Para cada serie y cada regresor, se busca una buena combinación de hiperparámetros con `bayesian_search_forecaster`, validando con `TimeSeriesFold` (ventanas sucesivas de 24 horas) sobre el tramo de entrenamiento — sin tocar las últimas `HORAS_TEST` horas, que quedan reservadas como holdout final para la comparación justa contra SARIMA y VAR.

In [ ]:
MODELOS = {
    "LightGBM": (LGBMRegressor(random_state=123, verbose=-1), {
        "n_estimators": ("int", 100, 500),
        "max_depth": ("int", 3, 8),
        "learning_rate": ("float", 0.01, 0.2),
    }),
    "XGBoost": (XGBRegressor(random_state=123), {
        "n_estimators": ("int", 100, 500),
        "max_depth": ("int", 3, 8),
        "learning_rate": ("float", 0.01, 0.2),
    }),
    "RandomForest": (RandomForestRegressor(random_state=123, n_jobs=-1), {
        "n_estimators": ("int", 100, 400),
        "max_depth": ("int", 4, 12),
    }),
}


def construir_espacio_busqueda(param_spec):
    def espacio(trial):
        params = {}
        for nombre, (tipo, low, high) in param_spec.items():
            if tipo == "int":
                params[nombre] = trial.suggest_int(nombre, low, high)
            else:
                params[nombre] = trial.suggest_float(nombre, low, high)
        return params
    return espacio

In [ ]:
resultados_ml = {}

for col in datos.columns:
    print(f"\n{'='*60}\n{col}\n{'='*60}")
    y_train_completo = datos_pos[col].iloc[:-HORAS_TEST]
    exog_train_completo = exog.iloc[:-HORAS_TEST]
    y_test = datos_pos[col].iloc[-HORAS_TEST:]
    exog_test = exog.iloc[-HORAS_TEST:]

    resultados_ml[col] = {}

    for nombre_modelo, (estimador, espacio_params) in MODELOS.items():
        forecaster = ForecasterRecursive(estimator=estimador, lags=LAGS)
        cv_busqueda = TimeSeriesFold(
            steps=24,
            initial_train_size=len(y_train_completo) - 24 * 60,  # últimos 60 días para validar
            refit=False,
        )
        resultados_busqueda, _ = bayesian_search_forecaster(
            forecaster=forecaster,
            y=y_train_completo,
            exog=exog_train_completo,
            cv=cv_busqueda,
            search_space=construir_espacio_busqueda(espacio_params),
            metric="mean_absolute_error",
            n_trials=15,
            return_best=True,
            verbose=False,
            show_progress=False,
        )
        mejores_params = resultados_busqueda.iloc[0]["params"]
        mae_validacion = resultados_busqueda.iloc[0]["mean_absolute_error"]
        print(f"  [{nombre_modelo}] mejores parámetros: {mejores_params}  (MAE validación={mae_validacion:.5f})")

        # return_best=True ya reentrenó `forecaster` con los mejores parámetros
        # sobre todo y_train_completo. Se pronostica directamente el holdout
        # final (168 horas nunca vistas) en un solo paso recursivo.
        pronostico_holdout = forecaster.predict(steps=HORAS_TEST, exog=exog_test)

        mae = mean_absolute_error(y_test.values, pronostico_holdout.values)
        rmse = mean_squared_error(y_test.values, pronostico_holdout.values) ** 0.5
        resultados_ml[col][nombre_modelo] = {"MAE": mae, "RMSE": rmse, "params": mejores_params}
        print(f"  [{nombre_modelo}] HOLDOUT final -> MAE={mae:.5f}  RMSE={rmse:.5f}")

## 3. Mejor modelo ML por serie, graficado

In [ ]:
fig, axes = plt.subplots(len(datos.columns), 1, figsize=(16, 12))

for ax, col in zip(axes, datos.columns):
    mejor_modelo = min(resultados_ml[col], key=lambda m: resultados_ml[col][m]["MAE"])
    print(f"{col}: mejor modelo ML = {mejor_modelo}  (MAE={resultados_ml[col][mejor_modelo]['MAE']:.5f})")

    y_train_completo = datos_pos[col].iloc[:-HORAS_TEST]
    exog_train_completo = exog.iloc[:-HORAS_TEST]
    y_test = datos_pos[col].iloc[-HORAS_TEST:]
    exog_test = exog.iloc[-HORAS_TEST:]

    estimador, _ = MODELOS[mejor_modelo]
    forecaster = ForecasterRecursive(estimator=estimador, lags=LAGS)
    forecaster.set_params(resultados_ml[col][mejor_modelo]["params"])
    forecaster.fit(y=y_train_completo, exog=exog_train_completo)
    pronostico = forecaster.predict(steps=HORAS_TEST, exog=exog_test)

    ax.plot(fechas_reales[-HORAS_TEST:], y_test.values, label="Real")
    ax.plot(fechas_reales[-HORAS_TEST:], pronostico.values, label=f"Pronóstico ({mejor_modelo})", linestyle="--")
    ax.set_title(f"Pronóstico ML — {col}")
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Comparación final: SARIMA vs VAR vs mejor modelo ML

Mismo período de test (últimas 168 horas) en los tres enfoques, para decidir cuál conviene usar en el informe final.

In [ ]:
resultados_sarima = json.loads((PROJECT_ROOT / "data/processed/resultados_sarima.json").read_text(encoding="utf-8"))
resultados_var = json.loads((PROJECT_ROOT / "data/processed/resultados_var.json").read_text(encoding="utf-8"))

filas_comparacion = []
for col in datos.columns:
    mejor_ml = min(resultados_ml[col], key=lambda m: resultados_ml[col][m]["MAE"])
    filas_comparacion.append({
        "serie": col,
        "SARIMA_MAE": resultados_sarima["metricas"][col]["MAE"],
        "SARIMA_RMSE": resultados_sarima["metricas"][col]["RMSE"],
        "VAR_MAE": resultados_var["metricas"][col]["MAE"],
        "VAR_RMSE": resultados_var["metricas"][col]["RMSE"],
        f"ML_mejor_modelo": mejor_ml,
        "ML_MAE": resultados_ml[col][mejor_ml]["MAE"],
        "ML_RMSE": resultados_ml[col][mejor_ml]["RMSE"],
    })

comparacion_final = pd.DataFrame(filas_comparacion).set_index("serie")
print(comparacion_final.to_string())

destino = PROJECT_ROOT / "data/processed/comparacion_final_modelos.csv"
comparacion_final.to_csv(destino)
print(f"\nGuardado en {destino}")
print("\nProyecto (notebooks 01 a 07) completo.")